## Imports e Configuração Inicial

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns

# Adiciona pasta src ao path
sys.path.append(os.path.abspath(os.path.join('..')))

from src.dataset import get_svhn_loaders
from src.model import SVHNNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Carrega os dados (batch_size maior ajuda a estabilizar gradientes para visualizar schedulers)
train_loader, test_loader = get_svhn_loaders(batch_size=128)

## Função de Treino com Registro de LR

Esta função é "híbrida": ela aceita um argumento step_on_batch para saber se deve atualizar a LR a cada iteração ou só no final da época.


In [ ]:
def train_with_scheduler(model, optimizer, scheduler, scheduler_name, epochs=10, step_on_batch=False):
    criterion = nn.CrossEntropyLoss()
    
    # Históricos
    lr_history = []
    loss_history = []
    val_acc_history = []
    
    print(f"--- Iniciando Treino com {scheduler_name} ---")
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            # Captura a LR atual
            current_lr = optimizer.param_groups[0]['lr']
            lr_history.append(current_lr)
            
            # Se o scheduler for por batch (ex: OneCycleLR, CyclicLR), atualiza aqui
            if step_on_batch:
                scheduler.step()
        
        # Se o scheduler for por época (ex: StepLR), atualiza aqui
        if not step_on_batch:
            scheduler.step()
            
        # Validação rápida
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        val_acc = 100 * correct / total
        val_acc_history.append(val_acc)
        avg_loss = running_loss / len(train_loader)
        loss_history.append(avg_loss)
        
        print(f"Epoch [{epoch+1}/{epochs}] LR: {current_lr:.6f} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.2f}%")
        
    return lr_history, loss_history, val_acc_history

## Experimento 1 - StepLR (O Clássico)
Aqui vamos usar o StepLR. Ele mantém a LR constante e a derruba drasticamente a cada X épocas. É ótimo para "refinar" o modelo quando ele para de aprender.

In [ ]:
# Setup Experimento 1: StepLR
model_step = SVHNNet().to(device)
# Usamos SGD com Momentum alto para ver bem o efeito
optimizer_step = optim.SGD(model_step.parameters(), lr=0.1, momentum=0.9)

# Decai a LR por um fator de 0.1 a cada 5 épocas
scheduler_step = optim.lr_scheduler.StepLR(optimizer_step, step_size=5, gamma=0.1)

# Executa (step_on_batch=False pois StepLR é por época)
lr_hist_step, loss_hist_step, acc_hist_step = train_with_scheduler(
    model_step, 
    optimizer_step, 
    scheduler_step, 
    "StepLR", 
    epochs=15, 
    step_on_batch=False
)

## Experimento 2 - OneCycleLR (O Moderno)
Aqui usamos o OneCycleLR. Ele começa com uma LR baixa, sobe até um máximo (aquecimento) e depois desce até quase zero. Isso geralmente converge muito mais rápido (Super-convergence).

In [ ]:
# Setup Experimento 2: OneCycleLR
model_cycle = SVHNNet().to(device)
optimizer_cycle = optim.SGD(model_cycle.parameters(), lr=0.01, momentum=0.9)

# Configuração do OneCycle
# max_lr: O pico que a LR vai atingir
# steps_per_epoch: Necessário saber quantos batches temos
# epochs: Total de épocas
scheduler_cycle = optim.lr_scheduler.OneCycleLR(
    optimizer_cycle, 
    max_lr=0.1, 
    steps_per_epoch=len(train_loader), 
    epochs=15
)

# Executa (step_on_batch=True pois OneCycleLR atualiza a cada iteração)
lr_hist_cycle, loss_hist_cycle, acc_hist_cycle = train_with_scheduler(
    model_cycle, 
    optimizer_cycle, 
    scheduler_cycle, 
    "OneCycleLR", 
    epochs=15, 
    step_on_batch=True
)

## Visualização Comparativa (Obrigatória para a Nota Técnica)

In [ ]:
# Configuração dos plots
fig, ax = plt.subplots(1, 3, figsize=(20, 5))

# 1. Evolução da Learning Rate
ax[0].plot(lr_hist_step, label='StepLR', color='blue')
# Precisamos ajustar a escala x do StepLR para bater com o OneCycle (que tem mais pontos) se quisermos sobrepor perfeitamente,
# mas plotar cru já mostra a diferença de comportamento (escada vs onda).
ax[0].plot(lr_hist_cycle, label='OneCycleLR', color='orange', alpha=0.7)
ax[0].set_title("Evolução da Learning Rate")
ax[0].set_xlabel("Steps (Batches)")
ax[0].set_ylabel("LR")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

# 2. Curvas de Loss
ax[1].plot(loss_hist_step, label='StepLR', marker='o')
ax[1].plot(loss_hist_cycle, label='OneCycleLR', marker='o')
ax[1].set_title("Training Loss por Época")
ax[1].set_xlabel("Época")
ax[1].set_ylabel("Loss")
ax[1].legend()
ax[1].grid(True)

# 3. Acurácia de Validação
ax[2].plot(acc_hist_step, label='StepLR', marker='o')
ax[2].plot(acc_hist_cycle, label='OneCycleLR', marker='o')
ax[2].set_title("Acurácia de Validação (%)")
ax[2].set_xlabel("Época")
ax[2].set_ylabel("Acurácia")
ax[2].legend()
ax[2].grid(True)

plt.tight_layout()
plt.show()